# SciBERT + SHAP-IG consensus


## Part A. Setup, data, training

In [ ]:
!pip install -q -U transformers accelerate shap captum lime nltk scipy scikit-learn seaborn


In [ ]:
import os, json, re, gc, pickle, random, shutil
import numpy as np, pandas as pd, torch
from collections import defaultdict
from google.colab import drive

if not os.path.ismount("/content/drive"):
    if os.path.exists("/content/drive"):
        shutil.rmtree("/content/drive")   # leftover local folder, not real Drive
    drive.mount("/content/drive")

ARXIV_PATH = "/content/drive/MyDrive/arxiv-metadata-oai-snapshot.json"
SAVE_ROOT  = "/content/drive/MyDrive/scibert_runs"
os.makedirs(SAVE_ROOT, exist_ok=True)

SEEDS = [42, 7]          
SPLIT_SEED = 42
TARGETS = ["cs.LG", "cs.CV", "cs.AI", "cs.CL", "cs.RO", "cs.CR", "cs.CY"]
PER_CLASS = 5000
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:

count, rows = defaultdict(int), []
seen_ids, seen_text = set(), set()
n_dup = 0

with open(ARXIV_PATH) as f:
    for line in f:
        p = json.loads(line)
        title, abstract = (p.get("title") or "").strip(), (p.get("abstract") or "").strip()
        if not title or not abstract:
            continue
        cats = p.get("categories", "").split()
        matched = [c for c in cats if c in TARGETS]
        if len(matched) != 1:
            continue
        cat = matched[0]
        if count[cat] >= PER_CLASS:
            continue
        text = re.sub(r"\s+", " ", title + " " + abstract)
        if p["id"] in seen_ids or text in seen_text:
            n_dup += 1
            continue
        seen_ids.add(p["id"]); seen_text.add(text)
        rows.append({"id": p["id"], "text": text, "category": cat,
                     "primary": cats[0],
                     "submitted": p["versions"][0]["created"]})
        count[cat] += 1
        if all(count[c] >= PER_CLASS for c in TARGETS):
            break

df = pd.DataFrame(rows)
df["submitted"] = pd.to_datetime(df["submitted"], utc=True)
print("size:", len(df), dict(count), "| duplicates skipped:", n_dup)


print("submitted between:", df.submitted.min().strftime("%B %Y"), "and", df.submitted.max().strftime("%B %Y"))
print("papers whose primary category is the labelled category:", int((df.primary == df.category).sum()), "of", len(df))

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])
class_names = [c.replace("cs.", "") for c in le.classes_]       # AI CL CR CV CY LG RO
SHORT = {"AI": "AI", "CL": "NLP", "CR": "CR", "CV": "CV", "CY": "CY", "LG": "ML", "RO": "RO"}
FULL = {"AI": "Artificial Intelligence", "CL": "Computation and Language",
        "CR": "Cryptography and Security", "CV": "Computer Vision",
        "CY": "Computers and Society", "LG": "Machine Learning", "RO": "Robotics"}
print(class_names)

tr_x, tmp_x, tr_y, tmp_y = train_test_split(df.text, df.label, test_size=0.3,
                                            stratify=df.label, random_state=SPLIT_SEED)
va_x, te_x, va_y, te_y = train_test_split(tmp_x, tmp_y, test_size=0.5,
                                          stratify=tmp_y, random_state=SPLIT_SEED)
print(len(tr_x), len(va_x), len(te_x))

tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")
enc = lambda x: tokenizer(list(x), truncation=True, padding=True, max_length=512)
tr_enc, va_enc, te_enc = enc(tr_x), enc(va_x), enc(te_x)

class ArxivDataset(torch.utils.data.Dataset):
    def __init__(self, e, y): self.e, self.y = e, list(y)
    def __getitem__(self, i):
        d = {k: torch.tensor(v[i]) for k, v in self.e.items()}
        d["labels"] = torch.tensor(self.y[i]); return d
    def __len__(self): return len(self.y)

train_ds, val_ds, test_ds = ArxivDataset(tr_enc, tr_y), ArxivDataset(va_enc, va_y), ArxivDataset(te_enc, te_y)
test_texts, test_labels = list(te_x), list(te_y)

In [ ]:
# ---- Train every seed (resumable: skips seeds already saved on Drive) ----
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    y, p = pred.label_ids, pred.predictions.argmax(-1)
    P, R, F, _ = precision_recall_fscore_support(y, p, average="weighted")
    return {"accuracy": accuracy_score(y, p), "f1": F, "precision": P, "recall": R}

for seed in SEEDS:
    d = f"{SAVE_ROOT}/seed{seed}"
    if os.path.exists(f"{d}/test_logits.npy"):
        print(f"seed {seed} already done"); continue
    print(f"\n===== SEED {seed} =====")
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        "allenai/scibert_scivocab_uncased", num_labels=len(class_names))
    args = TrainingArguments(
        output_dir=f"./results_seed{seed}", learning_rate=2e-5,
        per_device_train_batch_size=16, per_device_eval_batch_size=16,
        num_train_epochs=5, eval_strategy="epoch", save_strategy="epoch",
        save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="accuracy",
        fp16=True, label_smoothing_factor=0.1, seed=seed, data_seed=seed, report_to="none")
    tr = Trainer(model=model, args=args, train_dataset=train_ds,
                 eval_dataset=val_ds, compute_metrics=compute_metrics)
    tr.train()
    out = tr.predict(test_ds)
    os.makedirs(d, exist_ok=True)
    tr.save_model(d); tokenizer.save_pretrained(d)
    np.save(f"{d}/test_logits.npy", out.predictions)
    np.save(f"{d}/y_true.npy", out.label_ids)
    del model, tr, out; gc.collect(); torch.cuda.empty_cache()

## Part B. Classification results (Tables I and II, Fig. 2, confusion counts)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt

names = [SHORT[c] for c in class_names]
per_class, t1 = [], []
for seed in SEEDS:
    d = f"{SAVE_ROOT}/seed{seed}"
    y = np.load(f"{d}/y_true.npy"); p = np.load(f"{d}/test_logits.npy").argmax(1)
    P, R, F, _ = precision_recall_fscore_support(y, p, average="macro")
    t1.append({"Seed": seed, "Correct": int((y == p).sum()), "Accuracy": accuracy_score(y, p),
               "Macro-P": P, "Macro-R": R, "Macro-F1": F})
    per_class.append(pd.DataFrame(precision_recall_fscore_support(y, p), index=["P", "R", "F1", "n"],
                                  columns=names).T[["P", "R", "F1"]])
t1 = pd.DataFrame(t1)
print("TABLE I\n", t1.round(3).to_string(index=False))
print("Mean:", t1[["Accuracy", "Macro-P", "Macro-R", "Macro-F1"]].mean().round(3).to_dict())
t2 = sum(per_class) / len(per_class)
t2.loc["Macro average"] = t2.mean()
print("\nTABLE II (mean over seeds)\n", t2.round(3))

# Fig. 2 and confusion counts for the seed used in the explanations (first seed)
S0 = SEEDS[0]
y = np.load(f"{SAVE_ROOT}/seed{S0}/y_true.npy"); p = np.load(f"{SAVE_ROOT}/seed{S0}/test_logits.npy").argmax(1)
cm = confusion_matrix(y, p)
plt.figure(figsize=(10, 8), dpi=200)
ax = sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names,
                 linewidths=.7, linecolor="gray", annot_kws={"size": 14, "weight": "bold"})
plt.title(f"Confusion Matrix (seed {S0})", fontsize=18, fontweight="bold")
plt.xlabel("Predicted Label", fontsize=15, fontweight="bold"); plt.ylabel("True Label", fontsize=15, fontweight="bold")
plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=600, bbox_inches="tight"); plt.show()

pairs = sorted([(names[i], names[j], int(cm[i, j])) for i in range(7) for j in range(7) if i != j],
               key=lambda x: -x[2])[:8]
print("Top confusions (true -> predicted):", pairs)

## Part C. Explanations and fidelity (140 papers, seed-first model)

In [ ]:
import nltk
nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
from nltk.corpus import wordnet as wn
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as STOP

S0 = SEEDS[0]
tok = AutoTokenizer.from_pretrained(f"{SAVE_ROOT}/seed{S0}")
model = AutoModelForSequenceClassification.from_pretrained(f"{SAVE_ROOT}/seed{S0}").to(device).eval()

@torch.no_grad()
def predict_proba(texts, bs=32):
    texts, out = list(texts), []
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], truncation=True, padding=True, max_length=512, return_tensors="pt").to(device)
        out.append(torch.softmax(model(**e).logits.float(), -1).cpu().numpy())
    return np.vstack(out)

test_df = pd.DataFrame({"text": test_texts, "label": test_labels})
sample = test_df.groupby("label").sample(n=20, random_state=42).reset_index(drop=True)
print(len(sample), sample.label.value_counts().unique())

In [ ]:

import shap
from captum.attr import IntegratedGradients
from lime.lime_text import LimeTextExplainer

SPECIAL = {"[CLS]", "[SEP]", "[PAD]", "[UNK]"}

def merge_wordpieces(tokens, scores):
    # sum sub-word attributions into word-level scores
    words, vals = [], []
    for t, s in zip(tokens, scores):
        t = t.strip()
        if not t or t in SPECIAL: continue
        if t.startswith("##") and words:
            words[-1] += t[2:]; vals[-1] += float(s)
        else:
            words.append(t); vals.append(float(s))
    return words, vals

def top_candidates(words, vals, n=20):
    # positive attribution, no stop words, at least 3 letters, top-n by score
    agg = {}
    for w, v in zip(words, vals):
        w = w.lower(); agg[w] = agg.get(w, 0.0) + v
    keep = [(w, v) for w, v in agg.items()
            if v > 0 and len(w) >= 3 and re.fullmatch(r"[a-z]+", w) and w not in STOP]
    keep.sort(key=lambda x: -x[1])
    return [w for w, _ in keep[:n]]

shap_explainer = shap.Explainer(lambda x: predict_proba(list(x)), shap.maskers.Text(tok), algorithm="partition")
def shap_words(text, cls):
    sv = shap_explainer([text], max_evals=500, batch_size=32)
    return merge_wordpieces(list(sv.data[0]), sv.values[0][:, cls])

# IG: zero-vector baseline on the word embeddings, 50 steps
def _fwd(emb, mask, cls):
    return torch.softmax(model(inputs_embeds=emb, attention_mask=mask).logits.float(), -1)[:, cls]
ig = IntegratedGradients(_fwd)
def ig_words(text, cls):
    e = tok(text, truncation=True, max_length=512, return_tensors="pt").to(device)
    emb = model.bert.embeddings.word_embeddings(e["input_ids"]).detach()
    attr = ig.attribute(emb, baselines=torch.zeros_like(emb),
                        additional_forward_args=(e["attention_mask"], cls),
                        n_steps=50, internal_batch_size=10)
    scores = attr.sum(-1)[0].detach().cpu().numpy()            # sum over embedding dims
    return merge_wordpieces(tok.convert_ids_to_tokens(e["input_ids"][0]), scores)

# LIME: 1,000 perturbation samples, fixed seed 42
lime_explainer = LimeTextExplainer(class_names=class_names, random_state=42)
def lime_words(text, cls):
    ex = lime_explainer.explain_instance(text, predict_proba, labels=[cls], num_features=100, num_samples=1000)
    pairs = ex.as_list(label=cls)
    return [w for w, _ in pairs], [v for _, v in pairs]

# quick sanity check on one paper
t = sample.text[0]; c = int(predict_proba([t])[0].argmax())
print(top_candidates(*shap_words(t, c))[:10]); print(top_candidates(*ig_words(t, c))[:10])

In [ ]:

K = 7
def consensus(S, G, k=K):
    rS = {w: (i + 1) / 20 for i, w in enumerate(S)}
    rG = {w: (i + 1) / 20 for i, w in enumerate(G)}
    allw = list(dict.fromkeys(S + G))
    r = {w: (rS.get(w, 21/20) + rG.get(w, 21/20)) / 2 for w in allw}
    key = lambda w: (r[w], S.index(w) if w in rS else 99)      # ties -> better SHAP rank
    A = sorted([w for w in S if w in rG], key=key)
    if len(A) >= k: return A[:k]
    rest = sorted([w for w in allw if w not in A], key=key)
    return (A + rest)[:k]

def explain(text, cls=None):
    p = predict_proba([text])[0]
    cls = int(p.argmax()) if cls is None else cls
    cand = {"shap": top_candidates(*shap_words(text, cls)),
            "ig":   top_candidates(*ig_words(text, cls)),
            "lime": top_candidates(*lime_words(text, cls))}
    kws = {"shap": cand["shap"][:K], "ig": cand["ig"][:K], "lime": cand["lime"][:K],
           "consensus": consensus(cand["shap"], cand["ig"])}
    return cls, p, kws

def describe(pr): return "very high" if pr >= .95 else "high" if pr >= .80 else "moderate" if pr >= .60 else "low"
def make_explanation(cls, pr, kws):
    code_ = class_names[cls]
    return (f"This paper is classified as {FULL[code_]} (cs.{code_}) with {describe(pr)} confidence "
            f"({pr*100:.2f}%). The prediction is primarily influenced by domain-specific terms such as "
            f"{', '.join(kws)}. These terms contributed most to the model's classification decision.")

WORD = r"[A-Za-z]+"
def keep_only(text, kws):
    s = set(kws); return " ".join(w for w in re.findall(WORD, text) if w.lower() in s)
def remove(text, kws):
    s = set(kws); return re.sub(WORD, lambda m: "" if m.group(0).lower() in s else m.group(0), text)

def fidelity(text, cls, p0, kws):
    pk, pr = predict_proba([keep_only(text, kws)])[0], predict_proba([remove(text, kws)])[0]
    return {"suff": float(pk.argmax() == cls),            # class unchanged when only keywords kept
            "comp": float(p0[cls] - pr[cls]),              # probability drop when keywords removed
            "flip": float(pr.argmax() != cls)}             # class changes when keywords removed

def eligible_pool(text):
    return sorted({w for w in re.findall(WORD, text.lower()) if len(w) >= 3 and w not in STOP})

def synonym(w, rng):
    lem = {l.name().replace("_", " ") for s in wn.synsets(w) for l in s.lemmas()}
    lem = sorted(x for x in lem if x.isalpha() and x.lower() != w.lower())
    return rng.choice(lem) if lem else None

def perturb(text, protected, rng, frac=0.10):
    # replace 10% of non-keyword words with WordNet synonyms
    ms = [m for m in re.finditer(WORD, text)
          if m.group(0).lower() not in protected and len(m.group(0)) >= 3 and wn.synsets(m.group(0).lower())]
    n = max(1, round(frac * len(ms)))
    chosen = sorted(rng.sample(ms, min(n, len(ms))), key=lambda m: m.start())
    out, last = [], 0
    for m in chosen:
        s = synonym(m.group(0).lower(), rng)
        out += [text[last:m.start()], s if s else m.group(0)]; last = m.end()
    return "".join(out + [text[last:]])

In [ ]:

N_PERT = 5
METHODS = ["random", "lime", "shap", "ig", "consensus"]
CKPT = f"{SAVE_ROOT}/fidelity_rows.pkl"
rows = pickle.load(open(CKPT, "rb")) if os.path.exists(CKPT) else []
done = {r["paper"] for r in rows}

for i, rec in sample.iterrows():
    if i in done: continue
    text, rng = rec["text"], random.Random(1000 + i)
    cls, p0, kws = explain(text)
    pool = eligible_pool(text)
    protected = {w for m in ("shap", "ig", "lime", "consensus") for w in kws[m]}

    draws = [rng.sample(pool, min(K, len(pool))) for _ in range(20)]       # Random baseline
    fid = {"random": {k: float(np.mean([fidelity(text, cls, p0, d)[k] for d in draws]))
                      for k in ("suff", "comp", "flip")}}
    for m in ("lime", "shap", "ig", "consensus"):
        fid[m] = fidelity(text, cls, p0, kws[m])

    stab, kept = {m: [] for m in METHODS}, 0
    for _ in range(N_PERT):
        pt = perturb(text, protected, rng)
        if int(predict_proba([pt])[0].argmax()) != cls:    # keep only if prediction unchanged
            continue
        kept += 1
        _, _, kws2 = explain(pt, cls)
        for m in ("lime", "shap", "ig", "consensus"):
            stab[m].append(len(set(kws[m]) & set(kws2[m])) / K)
        pool2 = eligible_pool(pt)
        stab["random"].append(float(np.mean([len(set(d) & set(rng.sample(pool2, min(K, len(pool2))))) / K
                                             for d in draws])))
    for m in METHODS:
        rows.append({"paper": i, "method": m, **fid[m],
                     "stab": float(np.mean(stab[m])) if stab[m] else np.nan, "n_kept": kept,
                     "keywords": kws.get(m, draws[0]), 
                     "expl": make_explanation(cls, p0[cls], kws[m]) if m == "consensus" else ""})
    pickle.dump(rows, open(CKPT, "wb"))
    print(f"paper {i+1}/{len(sample)}", end="\r")

In [ ]:
# ---- Table III: means, bootstrap CIs, Wilcoxon with Bonferroni ----
from scipy.stats import wilcoxon
res = pd.DataFrame(rows)
M = {"suff": "Sufficiency", "comp": "Comprehensiveness", "flip": "Flip rate", "stab": "Stability"}
brng = np.random.default_rng(42)

def boot_ci(x, B=10000):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return np.percentile(x[brng.integers(0, len(x), (B, len(x)))].mean(1), [2.5, 97.5])

tab = []
for m in METHODS:
    r = {"Method": m}
    for k, n in M.items():
        x = res[res.method == m][k].values; lo, hi = boot_ci(x)
        r[n] = f"{np.nanmean(x):.3f} [{lo:.3f}, {hi:.3f}]"
    tab.append(r)
print("TABLE III\n", pd.DataFrame(tab).to_string(index=False))

ALPHA = 0.05 / 16
print(f"\nBonferroni alpha = {ALPHA:.4f}")
cons = res[res.method == "consensus"].set_index("paper")
for base in ["random", "lime", "shap", "ig"]:
    b = res[res.method == base].set_index("paper")
    for k, n in M.items():
        a, c = cons[k], b[k]; ok = a.notna() & c.notna()
        try: pv = wilcoxon(a[ok], c[ok]).pvalue
        except ValueError: pv = 1.0
        print(f"consensus vs {base:6s} | {n:17s} p={pv:.4f} {'SIG' if pv < ALPHA else 'ns'}")

print("\nperturbed copies kept per paper (of 5): mean", res[res.method=="consensus"].n_kept.mean().round(2),
      "| papers with none kept:", int((res[res.method=="consensus"].n_kept == 0).sum()))
print("\nExample explanation:\n", res[res.method == "consensus"].expl.iloc[0])

## Part D. User study statistics (Table IV)


In [ ]:
from scipy.stats import ttest_rel, wilcoxon
us = pd.read_csv("/content/drive/MyDrive/user_study_long.csv")
items = ["Confidence", "Reliability", "Trust", "Suspicion", "Efficiency", "Adoption Intent", "Understanding"]

def paired(a, b):
    ok = a.notna() & b.notna(); a, b = a[ok], b[ok]
    t, p = ttest_rel(b, a); diff = b - a
    try: pw = wilcoxon(a, b).pvalue
    except ValueError: pw = 1.0
    return dict(N=len(a), M_wo=a.mean(), M_with=b.mean(), t=t, p=p, d=diff.mean() / diff.std(ddof=1), p_wilcoxon=pw)

# overall: average each participant's ratings across the 3 articles
ov = us.groupby(["participant", "item"])[["without", "with"]].mean().reset_index()
overall = pd.DataFrame({it: paired(ov[ov.item == it].set_index("participant")["without"],
                                   ov[ov.item == it].set_index("participant")["with"]) for it in items}).T
overall["sig_bonf"] = overall.p < 0.05 / 7
print("TABLE IV (overall)\n", overall.round(3))

art = {}
for a in us.article.unique():
    for it in items:
        s = us[(us.article == a) & (us.item == it)].set_index("participant")
        art[(a, it)] = paired(s["without"], s["with"])
art = pd.DataFrame(art).T
art["sig_bonf"] = art.p < 0.05 / 21
print("\nArticle-level significant:", int(art.sig_bonf.sum()), "of", len(art))
print(art[~art.sig_bonf].round(4))